This notebook currently requires the feature-dems-from-errorgens branch of pyGSTi. Note that the branch is under active development to generalize the code and integrate it into pyGSTi. So this notebook might break in the future!

In [3]:
import numpy as np
import stim
import pygsti
import pymatching

#Error generator propagation
from pygsti.errorgenpropagation.errorpropagator import ErrorGeneratorPropagator

#DEM construction
from pygsti.extras.dem_construction import pygsti_object_builders as obj
from pygsti.extras.dem_construction import dem_tools as dems

In [4]:
#This constructs a family of error models to study.
#Note that right now, SPAM error is limited to uniform single-qubit error, and the error on mid-circuit measurements is modelled by 
#(1) the Mdefault error, then (2) the perfect measurement, with reset, then (3) the rho0 error. 
def make_sh_sweep_error_param_dict(p=1, h_scale=0, spam_error=0):
    error_rates_dict = {}
    error_rates_dict['Gh'] = {('H','Z'):np.sqrt(0.0001*p*h_scale),('H','X'):np.sqrt(0.0001*p*h_scale),
                              ('S','Z'):0.0001*p+0.0001*p*(1-h_scale),('S','Y'):0.0001*p,('S','X'):0.0001*p*(1-h_scale)}
    error_rates_dict['Gcnot'] = {('H','IX'): np.sqrt(0.01*p*h_scale),('H', 'ZI'):np.sqrt(0.01*p*h_scale), ('H', 'ZX'):np.sqrt(0.01*p*h_scale),
                                 ('S','IX'): 0.01*p/15+0.01*p*(1-h_scale),('S', 'IY'):0.01*p/15,('S', 'IZ'):0.01*p/15,
                                  ('S', 'XX'):0.01*p/15,('S', 'XY'):0.01*p/15,('S', 'XZ'):0.01*p/15,
                                   ('S', 'YX'):0.01*p/15, ('S', 'YY'):0.01*p/15, ('S', 'YZ'):0.01*p/15,
                                  ('S', 'ZX'):0.01*p/15+0.01*p*(1-h_scale), ('S', 'ZY'):0.01*p/15, ('S', 'ZZ'):0.01*p/15, 
                                  ('S','XI'): 0.01*p/15,('S', 'YI'):0.01*p/15,('S', 'ZI'):0.01*p/15+0.01*p*(1-h_scale)}
    error_rates_dict['Mdefault'] = {('S', 'X'): spam_error}
    error_rates_dict['rho0'] = {('S', 'X'): 0}

    return error_rates_dict

In [5]:
#Helper function for dealing with repeat blocks in stim circuits
def unroll_repeat_blocks(circuit, add_2q_idles=False, add_measurement_idles=False):
    new_circuit = stim.Circuit()
    
    for operation in circuit:
        if operation.name == "REPEAT":
            n_repeats = operation.repeat_count  # Assuming the first arg is the repeat count
            for _ in range(n_repeats):
                # Append the operations that follow the repeat block
                for op in operation.body_copy():
                    if op.name == "REPEAT":
                        break  # Stop if we hit another repeat block
                    new_circuit.append_operation(op.name, op.targets_copy(), op.gate_args_copy())
                    if (op.name=='MR' or op.name=='R') and add_measurement_idles:
                        #add idles on unmeasured qubits
                        unmeasured_qubits = set([q for q in range(circuit.num_qubits)])-set(t.qubit_value for t in op.targets_copy())
                        #print(op, unmeasured_qubits)
                        new_circuit.append_operation('I', unmeasured_qubits, op.gate_args_copy())
                    elif (op.name=='CX' or op.name=='CZ') and add_2q_idles:
                        unused_qubits = set([q for q in range(circuit.num_qubits)])-set(t.qubit_value for t in op.targets_copy())
                        #print(op, unused_qubits)
                        new_circuit.append_operation('I', unused_qubits, op.gate_args_copy())
        else:
            # Append operations that are not part of a repeat block
            new_circuit.append_operation(operation.name, operation.targets_copy())
            if (operation.name=='MR' or operation.name=='R') and add_measurement_idles:
                unmeasured_qubits = set([q for q in range(circuit.num_qubits)])-set(t.qubit_value for t in operation.targets_copy())
                #print(operation, unmeasured_qubits)
                new_circuit.append_operation('I', unmeasured_qubits, operation.gate_args_copy())
            elif (operation.name=='CX' or operation.name=='CZ') and add_2q_idles:
                unused_qubits = set([q for q in range(circuit.num_qubits)])-set(t.qubit_value for t in operation.targets_copy())
                #print(operation, unused_qubits)
                new_circuit.append_operation('I', unused_qubits, operation.gate_args_copy())
    
    return new_circuit

In [6]:
#For demo purposes, I am constructing a DEM for a single model/code distance
d=5
p = 0.5
hscale = 0.5

In [7]:
rounds = d
error_rates_dict = make_sh_sweep_error_param_dict(p=p, h_scale=hscale, spam_error=0)

In [8]:
#Here, we use a stim circuit that has detectors specified in it
#you can replace this with any other stim circuit you'd like. The only caveat (as of 04/14/26, will probably be fixed shortly) is that there are some bugs 
#in circuits where the line OBSERVABLE_INCLUDE(n) for a fixed n occurs at multiple points in the circuit. 
stimc = stim.Circuit.generated("surface_code:rotated_memory_z", distance=d, rounds=rounds)

#deal with repeat blocks in stim
stimc = unroll_repeat_blocks(stimc)


In [9]:
#Sets up pyGSTi objects for DEM creation based on the stim circuit
#makre sure include_observables=True for doing logical error rate estimation
pcircuit, qubit_mapping, measurements, detectors = obj.stim_to_pygsti_circuit(stimc, range(stimc.num_qubits), qubit_relabelling_dict=None, show_qubit_mappings=False, include_idles=False, include_observables=True)
pspec = obj.create_processor_spec(pcircuit, pcircuit.line_labels, gates=['Gh','Gcnot']) 
n_qubits = len(pcircuit.line_labels)
model = obj.build_model(error_rates_dict, pspec, oneQ_gate_names=['Gh'], twoQ_gate_names=['Gcnot'])

In [10]:
#constructs the detectors as Paulis
dets_as_pauli_strings = [dems.get_detector_as_parity(d, measurements, n_qubits) for d in detectors]

In [11]:
#sets up a stim Tableau, needed for calculating DEM event contributions
pcircuit = pcircuit.serialize()
stim_c_no_extras = obj.pygsti_c_to_stim(pcircuit)
tableau = stim.Tableau.from_circuit(stim_c_no_extras, ignore_measurement=True) 
inverse_tableau = tableau.inverse()
sim = stim.TableauSimulator()
sim.set_inverse_tableau(inverse_tableau)

In [12]:
#propagates errors and combines them into circuit error channel
egp = ErrorGeneratorPropagator(model)
eoc_eeg = egp.propagate_errorgens_bch(pcircuit, bch_order=1, include_spam=True)

In [13]:
#constructs the DEM
total_dem = dems.generate_dem_higher_order(dets_as_pauli_strings, eoc_eeg, sim, zassenhaus_order=1, add_type='compose')
#the previous line represents the DEM as a dictionary. Now construct a Stim DEM
stim_dem_str = dems.format_dem_stim(total_dem, n_logical=1)
stim_dem = stim.DetectorErrorModel(stim_dem_str)

In [14]:
#Basic pymatching decoding pipeline
n_shots = 100000
dec = pymatching.Matching.from_detector_error_model(stim_dem)

sampler = stim_dem.compile_sampler()
samples = sampler.sample(shots=n_shots)[0]

syndrome, log_flips, _ = sampler.sample(shots=n_shots)

predictions = dec.decode_batch(syndrome)
log_errors = (predictions != log_flips)

ler = sum(log_errors)/n_shots
print(f'LER of {ler[0]} for d={d}, p={p}')

LER of 0.10993 for d=5, p=0.5
